# ComplaintIQ - supervised EDA (`02_eda_supervised`)

EDA for supervised tasks: predicting `monetary_relief`, ranking complaints, and product/theme classification. `01_explore.ipynb` is the lightweight sanity check. Unsupervised EDA is `03_eda_unsupervised.ipynb`. EDA only, no modeling.

Both EDA notebooks share a data foundation, then diverge by task. The ETL emits the documented 18-column snake_case schema. Target and text features are derived, so this notebook validates them.

## EDA objectives

Explore to make defensible modeling choices. Each section produces evidence for a downstream decision:

- Trust the data. Confirm the ETL matches the schema. Quantify duplicates and missingness.
- Choose the metric. Measure how imbalanced `monetary_relief` is.
- Choose the features. See which inputs carry signal. Identify post-resolution leakage.
- Choose the split. Check if the target drifts over time. Requires time-based split if it does.
- Choose text parameters. Size the vocabulary and narrative length for TF-IDF.

> **Note:** Spark reads the Parquet and runs every aggregation on the full dataset. We only `.toPandas()` small results or fixed-seed samples.

> **Go deeper:**
> - [CFPB Consumer Complaint Database](https://www.consumerfinance.gov/data-research/consumer-complaints/): source dataset and collection.
> - [PySpark: DataFrame quickstart](https://spark.apache.org/docs/latest/api/python/getting_started/quickstart_df.html): Spark operations used here (~10 min).

## Prerequisite: regenerate the Parquet first

The file on disk may be stale. Regenerate: `make parquet` (full) or `make parquet NARRATIVE_ONLY=1`. `make validate-parquet` runs after; the schema-check cell below also asserts the contract.

## How to read this notebook

Written to be **reproducible** and followed by newcomers. Columns are loaded by explicit name, samples use fixed `RANDOM_STATE`, and each figure names the decision it drives.

> **Note:** a fact to pin down.
>
> **Tip:** a good habit.
>
> **Warning:** something that would bias a downstream model if ignored.
>
> **Go deeper:** curated background link.

Each figure carries a **What you're seeing / Notice / Why it matters** caption, where *Why it matters* states the concrete modeling decision the evidence drives.

## Setup

Same imports and house theme as `01_explore` and `03_eda_unsupervised`.

In [ ]:
import sys
from pathlib import Path

# Import shared helpers from the local complaintiq package. Walk up from the cwd
# to find the repo's src/ dir, so this works whether the notebook lives in
# notebooks/ or notebooks/appendix/, locally or in a Databricks Git folder.
_here = Path.cwd()
_src = None
for _p in [_here, *_here.parents]:
    if (_p / "src" / "complaintiq").exists():
        _src = str(_p / "src")
        break
if _src and _src not in sys.path:
    sys.path.insert(0, _src)

In [ ]:
from __future__ import annotations
from collections.abc import Iterable
from complaintiq import RANDOM_STATE, np, pd, plt, sns, print_versions  # shared setup
from complaintiq.text import STOPWORDS, tokenize, top_document_terms, top_bigrams

# The tools we lean on across the project.
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# House plotting theme: clean grid + colorblind-safe palette, used everywhere.
sns.set_theme(style="whitegrid", palette="colorblind", context="notebook", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 6)

# The one true seed, so any sample below is reproducible run-to-run.
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("python :", sys.version.split()[0])
print("numpy  :", np.__version__)
print("pandas :", pd.__version__)
print("seaborn:", sns.__version__)

In [ ]:
import re
from collections import Counter

# A tiny stoplist keeps the token views readable without pulling in a modeling
# library. This is descriptive counting (EDA), not feature extraction.

---
## 1. Load + schema check *(shared foundation)*

EDA works against the **full** dataset so the target balance is exact. Spark reads
the Parquet lazily and computes the aggregations without materializing it; `complaint_text`
is the one heavy column, left out of the projection here (Section 12 loads a text *sample*).
We lean on the derived `has_narrative` / `complaint_text_length` instead.

> **Tip (Spark):** select only the columns you need so Spark can prune them at the scan,
> and let the aggregations run on the cluster - pull results to pandas only to plot.

In [ ]:
from pathlib import Path
from pyspark.sql import functions as F

# Resolve the data directory: the UC volume when running on Databricks,
# else the local ../data produced by `make parquet`.
VOLUME_DIR = Path("/Volumes/workspace/complaintiq/data")
data_dir = VOLUME_DIR if VOLUME_DIR.exists() else Path("..") / "data"
path = data_dir / "complaints.parquet"
if not path.exists():
    raise FileNotFoundError("data/complaints.parquet not found. Run `make parquet` first.")

EXPECTED_COLUMNS = [
    "complaint_id",
    "date_received",
    "date_sent_to_company",
    "product",
    "sub_product",
    "issue",
    "sub_issue",
    "complaint_text",
    "company",
    "state",
    "zip_code",
    "tags",
    "submitted_via",
    "timely_response",
    "company_response_to_consumer",
    "complaint_text_length",
    "has_narrative",
    "monetary_relief",
]

file_cols = spark.read.parquet(str(path)).columns
missing = set(EXPECTED_COLUMNS) - set(file_cols)
unexpected = set(file_cols) - set(EXPECTED_COLUMNS)
assert not missing and not unexpected, (
    f"Parquet columns don't match schema.h. Missing: {missing}; unexpected: {unexpected}. "
    "Regenerate with `make parquet`."
)
print(f"Schema OK: {len(file_cols)} columns match schema.h")

load_cols = [c for c in EXPECTED_COLUMNS if c != "complaint_text"]
df = spark.read.parquet(str(path)).select(*load_cols)
n_rows = df.count()
print(f"Loaded {n_rows:,} rows x {len(df.columns)} columns (complaint_text left on disk).")

---
## 2. Inspect: first pass *(shared foundation)*

Peek at the rows, then types, a numeric summary, and structure. Spark computes the
summary; the small preview and summary are pulled into pandas to display.

> **Go deeper:**
> - [PySpark: DataFrame.describe](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.describe.html): *the per-column numeric summary Spark computes over the full dataset, ~3 min.*

In [ ]:
print("First 5 rows:")
display(df.limit(5).toPandas())
print("\nData types:")
print(df.dtypes)
print("\nNumeric summary:")
display(df.describe().toPandas())
print("\nStructure:")
df.printSchema()

---
## 3. Data quality *(shared foundation)*

Integrity checks that matter regardless of the downstream task.

In [ ]:
# Data quality checks that matter to any downstream task (supervised or unsupervised).
print("rows:", f"{n_rows:,}")
print("duplicate complaint_id values:", n_rows - df.select("complaint_id").distinct().count())
print("fully duplicated rows:        ", n_rows - df.distinct().count())

# A zip_code should map to a single state; count zips seen with more than one.
zip_state_counts = (
    df.dropna(subset=["zip_code", "state"])
    .groupBy("zip_code")
    .agg(F.countDistinct("state").alias("n_states"))
)
n_zip_multi = zip_state_counts.filter(F.col("n_states") > 1).count()
n_zip_total = zip_state_counts.count()
print(f"\nzip_codes tied to >1 state: {n_zip_multi:,} of {n_zip_total:,} distinct zips")

# High-cardinality tail: how concentrated / how many singletons?
company_counts = df.groupBy("company").count()
n_companies = company_counts.count()
top_company_count = company_counts.agg(F.max("count")).first()[0]
n_singletons = company_counts.filter(F.col("count") == 1).count()
print(f"\ncompanies: {n_companies:,} distinct")
print(f"  top company = {top_company_count / n_rows:.2%} of rows")
print(f"  companies appearing exactly once: {n_singletons:,}")

> **What you're seeing:** integrity checks looking for duplicate keys/rows, whether a zip maps
> to one state, and how concentrated the high-cardinality `company` field is.
>
> **Notice:** watch for duplicate rows, zip/state inconsistencies, and a long tail of
> companies seen only once.
>
> **Why it matters:** duplicates must be resolved before splitting or clustering (they
> bias both); zip/state inconsistency decides whether geo is usable; the cardinality
> tail decides how you encode or group rare categories.

---
## 4. Missingness *(shared foundation)*

Null fraction per column, computed as a Spark aggregation over the full dataset.

> **Note:** `complaint_text` was left out of the projection, so its coverage is read from
> `has_narrative` in section 8.

> **Go deeper:**
> - [PySpark: handling missing data (DataFrame.na)](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.na.html): *how Spark represents and handles nulls, and options beyond dropping rows, ~8 min.*

In [ ]:
missing = (
    df.select(
        [(F.count(F.when(F.col(c).isNull(), c)) / F.lit(n_rows)).alias(c) for c in df.columns]
    )
    .toPandas()
    .T[0]
    .sort_values(ascending=False)
)
print("Missing fraction by column:")
print(missing.round(4))
ax = sns.barplot(
    x=missing.values, y=missing.index, hue=missing.index, palette="colorblind", legend=False
)
ax.set_xlabel("fraction missing")
ax.set_ylabel("")
ax.set_title("Missingness by column")
plt.tight_layout()
plt.show()

---
## 5. Target validation *(supervised)*

Audit the ETL's `monetary_relief`: it should be `1` exactly for `"Closed with monetary
relief"` and **not** for `"Closed with non-monetary relief"` (the substring trap).

In [ ]:
resp = F.lower(F.trim(F.col("company_response_to_consumer").cast("string")))
recomputed = (resp == "closed with monetary relief").cast("int")
agree = df.select(F.mean((recomputed == F.col("monetary_relief")).cast("int")).alias("a")).first()[
    "a"
]
print(f"ETL monetary_relief agrees with a fresh recompute on {agree:.4%} of rows")
print("\nmonetary_relief mean + count by response value:")
print(
    df.groupBy("company_response_to_consumer")
    .agg(F.mean("monetary_relief").alias("mean"), F.count(F.lit(1)).alias("size"))
    .orderBy(F.col("size").desc())
    .toPandas()
    .set_index("company_response_to_consumer")
)
n_bad = df.filter(resp.contains("non-monetary") & (F.col("monetary_relief") != 0)).count()
assert n_bad == 0, "non-monetary-relief rows must map to monetary_relief == 0"
print("\nCheck passed: 'Closed with non-monetary relief' rows are all labeled 0.")

---
## 6. Target balance *(supervised)*

The label is a one-vs-rest collapse of `company_response_to_consumer`, expected to be
strongly imbalanced. The exact positive rate below decides the evaluation metric.

> **Note:** on the July 2026 snapshot the base rate is **~1.28%** (about 1 in 78
> complaints ends in monetary relief). The magnitude matters: a "3-5x lift" queue means
> something very different at a 1.28% base rate than at 15%, so the base rate is the
> denominator every lift and per-analyst-week number below is quoted against.

> **Go deeper:**
> - [scikit-learn: average_precision_score (PR-AUC)](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html): *why precision-recall AUC beats accuracy on a rare positive class, ~5 min.*
> - [imbalanced-learn](https://imbalanced-learn.org/stable/): *resampling and class-weight tools for skewed targets, ~10 min.*

In [ ]:
counts = (
    df.groupBy("monetary_relief")
    .count()
    .orderBy("monetary_relief")
    .toPandas()
    .set_index("monetary_relief")["count"]
)
print("monetary_relief counts:")
print(counts)
pos_rate = df.agg(F.mean("monetary_relief")).first()[0]
print(f"\npositive rate: {pos_rate:.4%}")
print("\nFull company_response_to_consumer distribution:")
resp_dist = (
    df.groupBy("company_response_to_consumer")
    .count()
    .toPandas()
    .set_index("company_response_to_consumer")["count"]
)
display((resp_dist / resp_dist.sum()).sort_values(ascending=False).round(4))

In [ ]:
ax = sns.barplot(
    x=counts.index.astype(str),
    y=counts.values,
    hue=counts.index.astype(str),
    palette="colorblind",
    legend=False,
)
ax.set_xlabel("monetary_relief")
ax.set_ylabel("complaints")
ax.set_title("Target is heavily imbalanced: monetary relief is rare")
for p in ax.patches:
    ax.annotate(
        f"{int(p.get_height()):,}",
        (p.get_x() + p.get_width() / 2, p.get_height()),
        ha="center",
        va="bottom",
        fontsize=11,
    )
plt.tight_layout()
plt.show()

> **What you're seeing:** complaint counts per class (printed values show the scale gap).
>
> **Notice:** the positive class is a small fraction of complaints (~1.28% on this snapshot).
>
> **Why it matters:** the measured positive rate is the input to the metric decision.
> Use PR-AUC and class weighting/resampling, and never report raw accuracy. It also
> sets the value math: an analyst reviewing ~500 complaints/week who works a top-ranked
> queue at Nx lift catches about `N x 1.28% x 500` relief cases/week (e.g. ~32/week at
> 5x vs ~6/week reading in arrival order), which is how the lift bar maps to hours saved.

---
## 7. Categorical breakdowns *(supervised)*

Cardinality, then the `product` mix. High-cardinality fields (`company`, `zip_code`,
`issue`) force encoding decisions.

In [ ]:
cat_cols = [
    "product",
    "sub_product",
    "issue",
    "sub_issue",
    "state",
    "submitted_via",
    "tags",
    "company",
    "zip_code",
]
print("Distinct values per categorical (cardinality):")
card = df.agg(*[F.countDistinct(c).alias(c) for c in cat_cols]).toPandas().T[0]
print(card.sort_values(ascending=False))

In [ ]:
print("Top products by complaint volume:")
prod_counts = df.groupBy("product").count().orderBy(F.col("count").desc()).limit(10).toPandas()
display(prod_counts.set_index("product")["count"])
ax = sns.barplot(
    data=prod_counts, x="count", y="product", hue="product", palette="colorblind", legend=False
)
ax.set_title("Complaint volume by product (top 10)")
ax.set_xlabel("complaints")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

> **What you're seeing:** the ten highest-volume products.
>
> **Notice:** volume concentrates in a few products.
>
> **Why it matters:** decides which categoricals are worth one-hot encoding directly and
> which need rare-category grouping (their rates are noisier, see section 10).

---
## 8. Narrative coverage *(supervised)*

Only a fraction of complaints carry a narrative (consent-gated), so text features reach a
subset. Quantify coverage and length.

> **Note:** coverage is **~23%** on the July 2026 snapshot, so ~77% of complaints have no
> narrative and can only be scored by the metadata-only fallback. Because both models feed
> **one** ranked queue, their scores must be **calibrated onto the same probability scale**
> (e.g. Platt / isotonic) and the top slice checked so it is not quietly dominated by one
> group; otherwise the merged ranking is comparing scores that do not mean the same thing.

In [ ]:
has_nar_rate = df.agg(F.mean(F.col("has_narrative").cast("int"))).first()[0]
print(f"has_narrative rate: {has_nar_rate:.4%}")
hn_counts = (
    df.groupBy("has_narrative")
    .count()
    .orderBy(F.col("count").desc())
    .toPandas()
    .set_index("has_narrative")["count"]
)
print(hn_counts)
lengths = (
    df.filter(F.col("has_narrative"))
    .select("complaint_text_length")
    .toPandas()["complaint_text_length"]
)
print("\ncomplaint_text_length (narrative rows only):")
display(lengths.describe())

In [ ]:
ax = sns.histplot(lengths, bins=50, log_scale=(True, False))
ax.set_xlabel("complaint_text_length (characters, log scale)")
ax.set_ylabel("complaints")
ax.set_title("Narrative length distribution (rows with a narrative)")
plt.tight_layout()
plt.show()

> **What you're seeing:** narrative length in characters (log x-axis).
>
> **Notice:** most are short with a long tail.
>
> **Why it matters:** decides text truncation / max-length and whether a text model can
> only apply to the ~23% narrative subset (the ~77% without one need a metadata-only path).
> The two paths only combine into one queue if both are calibrated to a shared probability
> scale, so a metadata-only score of 0.6 ranks against a narrative-model 0.6 fairly.

---
## 9. Signal preview: relief rate by feature *(supervised)*

Which inputs separate the classes: relief rate by feature, plus numeric correlation.
Relief-rate-by-`product` is not just signal here, it is the **product-bucket heuristic
baseline** the learned model has to beat: rank complaints by their product's historical
relief rate and review the top of that queue. The cell after the per-product rates computes
that baseline's own lift at top-1% / 5% / 10%, so the "beat the baseline" bar is a measured number rather
than an assumption.

In [ ]:
for col in ["product", "submitted_via", "has_narrative", "timely_response"]:
    rate = (
        df.groupBy(col)
        .agg(F.mean("monetary_relief").alias("rate"))
        .orderBy(F.col("rate").desc())
        .limit(10)
        .toPandas()
        .set_index(col)["rate"]
    )
    print(f"\nmonetary_relief rate by {col}:")
    print(rate.round(4))

In [ ]:
top = list(
    df.groupBy("product").count().orderBy(F.col("count").desc()).limit(10).toPandas()["product"]
)
rate = (
    df.filter(F.col("product").isin(top))
    .groupBy("product")
    .agg(F.mean("monetary_relief").alias("rate"))
    .orderBy(F.col("rate").desc())
    .toPandas()
    .set_index("product")["rate"]
)
ax = sns.barplot(
    x=rate.values,
    y=rate.index.astype(str),
    hue=rate.index.astype(str),
    palette="colorblind",
    legend=False,
)
ax.set_xlabel("monetary relief rate")
ax.set_ylabel("")
ax.set_title("Monetary-relief rate by product (top 10 by volume)")
for p in ax.patches:
    ax.annotate(
        f"{p.get_width():.3f}",
        (p.get_width(), p.get_y() + p.get_height() / 2),
        ha="left",
        va="center",
        fontsize=10,
    )
plt.tight_layout()
plt.show()

---
### 9a. Product-bucket baseline: measured lift at top-1% / 5% / 10%

The lift claim only means something against a computed baseline. Learn each product's
relief rate on an **older training window**, score the held-out recent months with it (a
leakage-safe chronological split, mirroring §11), then read the precision and lift of the
top slices (1% / 5% / 10%). These are the numbers the learned model must beat, not the raw base rate. top-10% lift is capped at 10x by construction, so smaller queues reveal ranking power it cannot show.

In [ ]:
# Product-bucket heuristic baseline, evaluated leakage-safe on a time split.
# Learn product relief rates on older complaints, score the most recent months, and
# measure the top-decile precision + lift. All aggregation in Spark; only the small
# ranked eval frame is pulled to pandas.

product_date_df = df.select(
    "product", "monetary_relief", F.to_date("date_received").alias("d")
).dropna(subset=["d"])
# Split older vs recent at the 80th percentile of the date. approxQuantile needs a
# numeric column, so rank on epoch-days.
product_date_df = product_date_df.withColumn("epoch", F.datediff(F.col("d"), F.lit("1970-01-01")))
cut = product_date_df.approxQuantile("epoch", [0.80], 0.001)[0]
train = product_date_df.filter(F.col("epoch") <= cut)
evl = product_date_df.filter(F.col("epoch") > cut)

# product relief rate learned on TRAIN only
prod_rate = train.groupBy("product").agg(F.mean("monetary_relief").alias("score"))
train_base = train.agg(F.mean("monetary_relief")).first()[0]

# score eval rows by their product's train rate (unseen product -> train base rate)
scored = evl.join(prod_rate, on="product", how="left").withColumn(
    "score", F.coalesce(F.col("score"), F.lit(train_base))
)

eval_base = scored.agg(F.mean("monetary_relief")).first()[0]
n_eval = scored.count()

# Lift at top-1% / 5% / 10%. Threshold at the (1-k) score quantile then filter -- distributed,
# no global sort (a Window.row_number over all eval rows would funnel them through one executor).
# top-10% lift is capped at 10x by construction (precision <= 1 => lift <= 1/k); smaller queues
# have higher ceilings and show ranking power the 10% slice cannot express.
quantiles = scored.approxQuantile("score", [0.99, 0.95, 0.90], 0.001)  # top 1% / 5% / 10% cutoffs
lifts = {}
for kfrac, thr in zip((0.01, 0.05, 0.10), quantiles):
    prec_k = scored.filter(F.col("score") >= thr).agg(F.mean("monetary_relief")).first()[0]
    lifts[kfrac] = (prec_k, (prec_k / eval_base if eval_base else float("nan")))

print(f"eval rows (recent months): {n_eval:,}")
print(f"eval-window base rate:      {eval_base:.4%}")
for kfrac in (0.01, 0.05, 0.10):
    p, l = lifts[kfrac]
    print(f"baseline top-{int(kfrac * 100):>2}% : precision={p:.4%}  lift={l:.2f}x")
print(
    "^ the bar the learned model must beat (top-10% is near its 10x ceiling; smaller queues higher)"
)

In [ ]:
num_cols = ["complaint_text_length", "has_narrative", "timely_response", "monetary_relief"]
dnum = df.select([F.col(c).cast("double").alias(c) for c in num_cols])
corr = pd.DataFrame(np.eye(len(num_cols)), index=num_cols, columns=num_cols)
for i, a in enumerate(num_cols):
    for b in num_cols[i + 1 :]:
        c = dnum.stat.corr(a, b)
        corr.loc[a, b] = c
        corr.loc[b, a] = c
ax = sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
ax.set_title("Correlation: numeric / boolean signals vs target")
plt.tight_layout()
plt.show()

> **What you're seeing:** relief rate across products and correlation among numeric signals.
>
> **Notice:** relief rate varies widely by product; numeric signals correlate weakly.
>
> **Why it matters:** ranks candidate features, `product` and text earn their place;
> weakly-correlated numeric columns are low priority.

---
## 10. Bivariate signal & subgroup support *(supervised)*

Does each product carry enough positives to trust its rate, and does relief rate shift across a `product x submitted_via` interaction?

In [ ]:
support = (
    df.groupBy("product")
    .agg(
        F.count(F.lit(1)).alias("n"),
        F.sum("monetary_relief").alias("positives"),
        F.mean("monetary_relief").alias("rate"),
    )
    .orderBy(F.col("n").desc())
    .toPandas()
    .set_index("product")
)
print("Per-product support (n, positives, relief rate):")
display(support.round(4))

In [ ]:
top = list(
    df.groupBy("product").count().orderBy(F.col("count").desc()).limit(6).toPandas()["product"]
)
pivot = (
    df.filter(F.col("product").isin(top))
    .groupBy("product")
    .pivot("submitted_via")
    .agg(F.mean("monetary_relief"))
    .toPandas()
    .set_index("product")
)
ax = sns.heatmap(pivot, annot=True, fmt=".3f", cmap="coolwarm", center=0)
ax.set_title("Relief rate by product x submitted_via (top products)")
plt.tight_layout()
plt.show()

> **What you're seeing:** per-product positive counts and the product/channel interaction.
>
> **Notice:** some products have very few positives; channel can shift the rate.
>
> **Why it matters:** thin subgroups justify regularization / grouping rare categories,
> and the interaction warns against assuming features are independent.

---
## 11. Temporal EDA *(supervised)*

Does volume trend and does the relief **rate** drift? Drift is why you split by *time*.

> **Warning (leakage):** `date_sent_to_company` is intake metadata, but a feature built
> from it *and* the resolution (days-to-respond) would leak.

> **Go deeper:**
> - [scikit-learn: TimeSeriesSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html): *how to split by time so evaluation reflects deployment on future data, ~5 min.*

In [ ]:
monthly = (
    df.select(F.to_date("date_received").alias("d"), "monetary_relief")
    .dropna(subset=["d"])
    .withColumn("month", F.trunc("d", "month"))
    .groupBy("month")
    .agg(F.count(F.lit(1)).alias("volume"), F.mean("monetary_relief").alias("relief_rate"))
    .orderBy("month")
    .toPandas()
    .set_index("month")
)
print("Last few months:")
display(monthly.tail())
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.lineplot(x=monthly.index, y=monthly["volume"], ax=axes[0])
axes[0].set_title("Complaint volume by month")
axes[0].set_ylabel("complaints")
axes[0].set_xlabel("month")
sns.lineplot(x=monthly.index, y=monthly["relief_rate"], ax=axes[1])
axes[1].set_title("Monetary-relief rate by month")
axes[1].set_ylabel("relief rate")
axes[1].set_xlabel("month")
plt.tight_layout()
plt.show()

> **What you're seeing:** monthly volume (left) and monthly relief rate (right).
>
> **Notice:** volume grows; the relief rate is not flat.
>
> **Why it matters:** a drifting rate mandates a time-based split (train older, validate
> newer) so the reported score is reproducible on future complaints.

---
## 12. Text signal vs target *(supervised)*

Does narrative length differ by class, and which terms/bigrams skew toward relief? On a
fixed-seed **sample** (all narratives are unnecessary here).

> **Note:** corpus-wide vocabulary/sparsity lives in `03_eda_unsupervised.ipynb`.

> **Go deeper:**
> - [scikit-learn: TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html): *the min_df / max_df / max_features knobs this EDA is sizing, ~8 min.*

In [ ]:
nar_path = data_dir / "complaints_narrative_only.parquet"
if nar_path.exists():
    txt_sdf = spark.read.parquet(str(nar_path)).select(
        "complaint_text", "complaint_text_length", "monetary_relief"
    )
else:
    print("narrative-only file not found; sampling complaint_text from the full Parquet.")
    txt_sdf = (
        spark.read.parquet(str(path))
        .filter(F.col("has_narrative"))
        .select("complaint_text", "complaint_text_length", "monetary_relief")
    )
txt_sdf = txt_sdf.dropna(subset=["complaint_text"])
n_txt = txt_sdf.count()
SAMPLE_N = min(n_txt, 20000)
# Stratify on the target: monetary relief is rare, so a uniform sample can miss
# it. Draw an exact per-class quota via a groupby + window: count each class,
# take its proportional share, then keep that many rows ranked in random order
# within the class. Proportional (not equal) keeps the true positive rate the
# EDA reports, while guaranteeing the rare class is represented.
from pyspark.sql import Window

frac = min(1.0, SAMPLE_N / n_txt) if n_txt else 0.0
class_counts = {
    r["monetary_relief"]: r["count"] for r in txt_sdf.groupBy("monetary_relief").count().collect()
}
per_class = {c: int(round(n * frac)) for c, n in class_counts.items()}
_rn = F.row_number().over(Window.partitionBy("monetary_relief").orderBy(F.rand(RANDOM_STATE)))
_cond = F.lit(False)
for _c, _k in per_class.items():
    _cond = _cond | ((F.col("monetary_relief") == F.lit(_c)) & (F.col("_rn") <= F.lit(_k)))
txt = txt_sdf.withColumn("_rn", _rn).filter(_cond).drop("_rn").toPandas()
print(f"per-class sample sizes: {per_class}")
print(f"Text sample: {len(txt):,} narratives (positive rate {txt['monetary_relief'].mean():.4%})")

In [ ]:
ax = sns.boxplot(
    data=txt,
    x="monetary_relief",
    y="complaint_text_length",
    hue="monetary_relief",
    palette="colorblind",
    legend=False,
)
ax.set_yscale("log")
ax.set_xlabel("monetary_relief")
ax.set_ylabel("complaint_text_length (log scale)")
ax.set_title("Narrative length by target class")
plt.tight_layout()
plt.show()

In [ ]:
for lab in (1, 0):
    docs = txt.loc[txt["monetary_relief"] == lab, "complaint_text"]
    print(f"\nmonetary_relief={lab}  (n={len(docs):,})")
    print("  top terms: ", [t for t, _ in top_document_terms(docs, k=10)])
    print("  top bigrams:", [b for b, _ in top_bigrams(docs, k=8)])

> **What you're seeing:** length by class, plus top terms and bigrams within each class.
>
> **Notice:** class term lists overlap on generic words; the *differences* are the signal.
>
> **Why it matters:** confirms the narrative is worth vectorizing and points at the
> stopword / `min_df` tuning that will make TF-IDF discriminative.

---
## 13. Leakage audit *(supervised)*

Post-resolution fields must be excluded as inputs:

- `company_response_to_consumer` - the label source; hard exclude.
- `timely_response` - known only after the company responds; treat as leakage.
- `date_sent_to_company` - intake metadata is fine, resolution-derived features leak.

Everything else is known at intake and is fair game.

In [ ]:
LEAKAGE = ["company_response_to_consumer", "timely_response"]
TARGET = "monetary_relief"
IDENTIFIERS = ["complaint_id"]
candidate_features = [c for c in EXPECTED_COLUMNS if c not in LEAKAGE + [TARGET] + IDENTIFIERS]
print("Candidate intake-time features:")
for c in candidate_features:
    print(" -", c)
print("\nTarget:              ", TARGET)
print("Excluded as leakage: ", LEAKAGE)
print("Identifiers dropped: ", IDENTIFIERS)

> **Decisions this EDA supports:**
> - **Data is trustworthy:** schema matches `schema.h`; keys, duplicates, and
>   missingness are quantified.
> - **Metric:** target is validated and rare (~1.28%): PR-AUC + class weighting/resampling.
> - **Baseline to beat:** the product-bucket heuristic already reaches ~9x lift at the
>   top-1% / 5% / 10% (measured in section 9a), so "useful" means beating that baseline, not the
>   raw base rate.
> - **Features:** `product` and narrative text carry signal; numeric columns are weak;
>   `company` / `zip_code` / `issue` need rare category handling.
> - **Split:** the target drifts over time: split by time, not at random.
> - **Text params:** narrative is sparse but discriminative: tune TF-IDF
>   `min_df` / `max_df` / `max_features`.
> - **Two-path scoring:** ~23% narrative (TF-IDF) vs ~77% metadata-only; calibrate both to
>   one probability scale and check the top slice is not dominated by one group.
> - **Leakage:** exclude `company_response_to_consumer` and `timely_response`.